# Fig. 3

In [1]:
import pandas as pd

In [2]:
df_raw = pd.read_csv("../dataset_1/raw_data/drugcomb/summary_v_1_5.csv", na_values=["\\N", "NA", "N/A"])
df_raw.dropna(subset=['drug_row', 'drug_col', 'cell_line_name'], inplace=True)
df_processed = pd.read_csv("../dataset_1/database/drug_comb/drugcomb_cleaned_with_mean_int.csv")
df_processed.dropna(subset=['drug_row', 'drug_col', 'cell_line_name'], inplace=True)

C:\Users\84991\AppData\Local\Temp\ipykernel_59716\2577098357.py:1: DtypeWarning: Columns (2,7,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../dataset_1/raw_data/drugcomb/summary_v_1_5.csv", na_values=["\\N", "NA", "N/A"])


In [17]:
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents

SCI_STYLE = {
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 900,
    'savefig.format': 'tiff',
    'axes.linewidth': 0.8, 
    'grid.linewidth': 0.5,  
}

def prepare_upset_data(df, filter_isolated=False):

    drug_combinations = df.groupby('study_name').apply(
        lambda g: set(g['drug_row'].dropna().astype(str)) | set(g['drug_col'].dropna().astype(str))
    )
    
    cell_line_combinations = df.groupby('study_name')['cell_line_name'].apply(
        lambda x: set(x.dropna().astype(str))
    )
    
    combo_combinations = df.groupby('study_name').apply(
        lambda g: set(zip(
            g['drug_row'].dropna().astype(str),
            g['drug_col'].dropna().astype(str),
            g['cell_line_name'].dropna().astype(str)
        ))
    )
    
    if filter_isolated:
        all_drugs = set()
        for drug_set in drug_combinations:
            all_drugs |= drug_set
        
        overlapping_drugs = set()
        drug_counts = {drug: 0 for drug in all_drugs}
        for drug_set in drug_combinations:
            for drug in drug_set:
                drug_counts[drug] += 1
        
        overlapping_drugs = {drug for drug, count in drug_counts.items() if count >= 2}
        
        filtered_drug_combinations = {}
        for study, drug_set in drug_combinations.items():
            filtered_drug_combinations[study] = drug_set & overlapping_drugs
        
        return filtered_drug_combinations, cell_line_combinations, combo_combinations
    
    return drug_combinations, cell_line_combinations, combo_combinations

def plot_full_width_upset(data, title, output_name, color='#1f77b4'):
    filtered_data = {k: v for k, v in data.items() if len(v) > 0}
    if not filtered_data: 
        return
    
    width, height = 7.5, 4.2
    
    fig = plt.figure(figsize=(width, height))
    plt.rcParams.update(SCI_STYLE)
    
    upset = UpSet(
        from_contents(filtered_data),
        facecolor=color,
        shading_color='#e6f2ff',
        subset_size='count',
        show_counts=True,
        sort_by='cardinality',
        sort_categories_by='cardinality',
        element_size=23
    )
    
    axes_dict = upset.plot(fig=fig)
    
    if 'shaded' in axes_dict:
        axes_dict['shaded'].invert_yaxis()
    
    if 'intersections' in axes_dict:
        ax = axes_dict['intersections']
        ax.grid(False)
    
    for key, ax in axes_dict.items():
        if key == 'shaded':
            continue 
            
        pos = ax.get_position()
        ax.set_position([
            pos.x0, 
            pos.y0, 
            pos.width,
            pos.height
        ])
        
        if key == 'intersections':
            ax.spines[['top', 'right']].set_visible(False)
        
        if key == 'intersections' and hasattr(ax, 'texts'):
            for text in ax.texts:
                x, y = text.get_position()
                text.set_position((x, y + 2))
                text.set_rotation(60)
    
    plt.savefig(
        f"{output_name}.tiff", 
        dpi=900,
        pad_inches=0.05
    )
    print(f"已保存: {output_name}.tiff")
    plt.close(fig)
    

def plot_half_width_upset(data, title, output_name, color='#1f77b4'):
    filtered_data = {k: v for k, v in data.items() if len(v) > 0}
    if not filtered_data: 
        return
    
    width, height = 3.5, 4.2
    
    fig = plt.figure(figsize=(width, height))
    plt.rcParams.update(SCI_STYLE)

    upset = UpSet(
        from_contents(filtered_data),
        facecolor=color,
        shading_color='#e6f2ff',
        subset_size='count',
        show_counts=True,
        sort_by='cardinality',
        sort_categories_by='cardinality',
        element_size=18
    )
    
    axes_dict = upset.plot(fig=fig)
    
    if 'shaded' in axes_dict:
        axes_dict['shaded'].invert_yaxis()
    
    if 'intersections' in axes_dict:
        ax = axes_dict['intersections']
        ax.grid(False)
    
    for key, ax in axes_dict.items():
        pos = ax.get_position()
        ax.set_position([
            pos.x0, 
            pos.y0, 
            pos.width * 0.90, 
            pos.height
        ])
        
        if key == 'intersections':
            ax.spines[['top', 'right']].set_visible(False)
        
        if key == 'intersections' and hasattr(ax, 'texts'):
            for text in ax.texts:
                text.set_rotation(60)
    
    plt.savefig(
        f"{output_name}.tiff", 
        dpi=900,
        pad_inches=0.05
    )
    print(f"已保存: {output_name}.tiff")
    plt.close(fig)


def filter_large_studies(df, min_entries=2000):
    study_counts = df['study_name'].value_counts()
    large_studies = study_counts[study_counts > min_entries].index.tolist()
    return df[df['study_name'].isin(large_studies)]

if __name__ == "__main__":
    df_processed_large = filter_large_studies(df_processed)
    drug_processed, cell_processed, combo_processed = prepare_upset_data(
        df_processed_large, 
        filter_isolated=True
    )
    
    plot_full_width_upset(
        drug_processed, 
        "Drug Overlaps - Processed Data", 
        "Drug_Overlaps_Processed",
        color='#1f77b4'
    )
    
    plot_half_width_upset(
        cell_processed, 
        "Cell Line Overlaps - Processed Data", 
        "Cell_Line_Overlaps_Processed",
        color='#1f77b4' 
    )
    
    plot_half_width_upset(
        combo_processed, 
        "Drug Combination Overlaps - Processed Data", 
        "Drug_Combo_Overlaps_Processed",
        color='#1f77b4'
    )

已保存: Drug_Overlaps_Processed.tiff
已保存: Cell_Line_Overlaps_Processed.tiff
已保存: Drug_Combo_Overlaps_Processed.tiff
